# Agent evaluation with MLflow

The traces in MLflow show what each agent did. This notebook scores the agents and stores the scores in MLflow,
so every release can be compared with the one before.

1. **Cash agent:** the June closure is planned 3 times. Each run must use the model, finish within 60 s and 8 tool
   calls, and propose what the engine allows: move the CHF-06 transfer to 30 July and a credit line of 20 000 €.
2. **Proof investigation:** the rent letter is uploaded once. The final label and amounts must match the reference
   case in `docs/PROOF.md` (conflict, 3 800,00 € justified, 304,00 € disputed per month). The run also shows whether
   the agent or the backend's fallback decided.

Each run becomes an MLflow run in the experiment `vero-agent-eval`, tagged with the app's run id, so it can be matched
with its trace in the experiment `vero-agent`.

This notebook posts events to the shared demo and calls `/demo/reset` before and after,
so do not run it while someone is presenting the app.

In [1]:
import time
import uuid
from pathlib import Path

import mlflow
import pandas as pd
import requests

API = "http://api.autonomous-cfo.svc.cluster.local:8080/api/v1"
MLFLOW = "http://mlflow.autonomous-cfo.svc.cluster.local:8080"
RUNS = 3
CLOSURE = {
    "type": "CLOSURE_PLANNED",
    "payload": {"date_from": "2025-06-01", "date_to": "2025-06-30", "variable_costs_stop": True, "note": "summer closure"},
}
HERO = Path.home() / "autonomous-cfo/services/api/src/api/fixtures/proof/pdf/doc_rent_adjustment_2025_06.pdf"
EXPECTED_CASH = {"move_ref": "CHF-06", "move_to": "2025-07-30", "credit_amount": "2000000"}
EXPECTED_PROOF = {"classification": "CONFLICT", "billed": "410400", "justified": "380000", "disputed": "30400"}


def get(path, **params):
    r = requests.get(f"{API}{path}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()


def post(path, body=None, files=None):
    r = requests.post(f"{API}{path}", json=None if files else (body or {}), files=files,
                      headers={"Idempotency-Key": str(uuid.uuid4())}, timeout=60)
    r.raise_for_status()
    return r.json()


def wait_run(run_id, limit=150):
    start = time.time()
    run = {}
    while time.time() - start < limit:
        run = get(f"/runs/{run_id}")
        if run["status"] in ("SUCCEEDED", "FAILED"):
            break
        time.sleep(1)
    return run, round(time.time() - start, 1)


MODEL = get("/health/model")["model"]
mlflow.set_tracking_uri(MLFLOW)
mlflow.set_experiment("vero-agent-eval")
print("model:", MODEL, "| mlflow client", mlflow.__version__)

2026/11/21 23:16:31 INFO mlflow.tracking.fluent: Experiment with name 'vero-agent-eval' does not exist. Creating a new experiment.


model: vero-llm | mlflow client 3.15.2


## 1. Cash agent: the June closure, 3 times

In [2]:
def cash_run():
    post("/demo/reset")
    version = get("/company")["financial_version"]
    accepted = post("/events", {**CLOSURE, "expected_financial_version": version})
    run, seconds = wait_run(accepted["run_id"])
    recs = [r for r in get("/recommendations") if r["run_id"] == run["run_id"]]
    move = next((r["subject"] for r in recs if r["kind"] == "RESCHEDULE_PAYMENT"), {})
    credit = next((r["subject"] for r in recs if r["kind"] == "CREDIT_LINE"), {})
    tools = [s["tool"] for s in run["trace"] if s["kind"] == "TOOL_CALL"]
    row = {
        "run_id": run["run_id"],
        "status": run["status"],
        "mode": run["mode"],
        "seconds": seconds,
        "tool_calls": len(tools),
        "proposals": len(recs),
        "explained_by_model": sum(1 for r in recs if (r.get("explanation") or {}).get("source") == "MODEL"),
        "move_ok": move.get("ref") == EXPECTED_CASH["move_ref"] and move.get("to_date") == EXPECTED_CASH["move_to"],
        "credit_ok": credit.get("amount") == EXPECTED_CASH["credit_amount"],
    }
    row["passed"] = (row["status"] == "SUCCEEDED" and row["mode"] == "AGENT" and row["seconds"] < 60
                     and row["tool_calls"] <= 8 and row["move_ok"] and row["credit_ok"])
    return row, tools


rows = []
for i in range(1, RUNS + 1):
    row, tools = cash_run()
    with mlflow.start_run(run_name=f"cash-closure-{i}"):
        mlflow.set_tags({"agent": "cash", "scenario": "june-closure", "app_run_id": row["run_id"],
                         "model": MODEL, "tools": ", ".join(tools)})
        mlflow.log_params({"max_tool_calls": 8, "deadline_s": 60})
        mlflow.log_metrics({k: float(row[k]) for k in
                            ("seconds", "tool_calls", "proposals", "explained_by_model", "move_ok", "credit_ok", "passed")})
    rows.append(row)

cash = pd.DataFrame(rows)
cash

🏃 View run cash-closure-1 at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2/runs/d56327375751494cba533f2a44539f02
🧪 View experiment at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2


🏃 View run cash-closure-2 at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2/runs/f0aae40e0133444a8279689d431a53b6
🧪 View experiment at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2


🏃 View run cash-closure-3 at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2/runs/54b8de29c4114677aca8beb601ddf1a4
🧪 View experiment at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2


,run_id,status,mode,seconds,tool_calls,proposals,explained_by_model,move_ok,credit_ok,passed
0,5a45356d-816b-41a5-8478-59dc5195e1df,SUCCEEDED,AGENT,22.3,4,2,2,True,True,True
1,a81b31cc-8af5-43c2-89b7-8afa94d5c36d,SUCCEEDED,AGENT,24.3,4,2,0,True,True,True
2,0756032b-ce95-4db7-9b0a-d23a7561ef33,SUCCEEDED,AGENT,22.3,4,2,2,True,True,True


## 2. Proof investigation: the rent letter

In [3]:
post("/demo/reset")
with HERO.open("rb") as f:
    accepted = post("/documents", files={"file": (HERO.name, f, "application/pdf")})
run, seconds = wait_run(accepted["run_id"])
finding = next(f for f in get("/findings") if f.get("document_id") == accepted["document_id"])
inv = finding.get("investigation") or {}
agent_tools = [s["tool"] for s in run["trace"] if s["kind"] == "TOOL_CALL" and s.get("tool")]
proof = {
    "run_id": run["run_id"],
    "status": run["status"],
    "mode": run["mode"],
    "seconds": seconds,
    "agent_tool_calls": len(agent_tools),
    "classification": finding["classification"],
    "billed": inv.get("billed"),
    "justified": inv.get("justified"),
    "disputed": inv.get("disputed"),
}
proof["decided_by_agent"] = run["mode"] == "AGENT"
proof["result_ok"] = all(proof[k] == v for k, v in EXPECTED_PROOF.items())

with mlflow.start_run(run_name="proof-rent-letter"):
    mlflow.set_tags({"agent": "investigation", "scenario": "rent-letter", "app_run_id": proof["run_id"],
                     "model": MODEL, "mode": proof["mode"], "tools": ", ".join(agent_tools)})
    mlflow.log_params({"max_tool_calls": 6, "deadline_s": 60})
    mlflow.log_metrics({"seconds": proof["seconds"], "agent_tool_calls": proof["agent_tool_calls"],
                        "decided_by_agent": float(proof["decided_by_agent"]), "result_ok": float(proof["result_ok"])})

pd.Series(proof)

🏃 View run proof-rent-letter at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2/runs/b93ad26e36d341148ffb9aeb210dc929
🧪 View experiment at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2


run_id              aee4a787-e2b8-4c7e-adf6-2076470d1c08
status                                         SUCCEEDED
mode                                               AGENT
seconds                                             41.6
agent_tool_calls                                       5
classification                               UNEXPLAINED
billed                                              None
justified                                           None
disputed                                            None
decided_by_agent                                    True
result_ok                                          False
dtype: object

## 3. Summary

One summary run per evaluation makes releases easy to compare in the MLflow UI
(`oc port-forward svc/mlflow 5000:8080`, then http://localhost:5000, experiment `vero-agent-eval`).

In [4]:
summary = {
    "cash_pass_rate": float(cash["passed"].mean()),
    "cash_mean_seconds": float(cash["seconds"].mean()),
    "cash_max_seconds": float(cash["seconds"].max()),
    "cash_mean_tool_calls": float(cash["tool_calls"].mean()),
    "proof_result_ok": float(proof["result_ok"]),
    "proof_decided_by_agent": float(proof["decided_by_agent"]),
    "proof_seconds": float(proof["seconds"]),
}
with mlflow.start_run(run_name="summary"):
    mlflow.set_tags({"model": MODEL, "runs": str(RUNS)})
    mlflow.log_metrics(summary)

post("/demo/reset")
pd.Series(summary)

🏃 View run summary at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2/runs/52e0c3157cee4e8cbec944b7cb8a7c92
🧪 View experiment at: http://mlflow.autonomous-cfo.svc.cluster.local:8080/#/experiments/2


cash_pass_rate             1.000000
cash_mean_seconds         22.966667
cash_max_seconds          24.300000
cash_mean_tool_calls       4.000000
proof_result_ok            0.000000
proof_decided_by_agent     1.000000
proof_seconds             41.600000
dtype: float64